# TA/HM 피처 중복 제거 5단계 실험

현재 `CatBoost + residual LSTM`을 기준으로 다섯 제거 단계를 누적 적용합니다. 2022~2024 rolling OOF 후 2025년 672행을 평가하고, 단일 시드 최고 후보는 시드 42~44로 다시 확인합니다.

In [ ]:
# 1. Drive 안전 마운트
from pathlib import Path
from google.colab import drive
import os

MOUNT_ROOT = Path('/content/drive')
if not os.path.ismount(MOUNT_ROOT):
    if MOUNT_ROOT.exists() and any(MOUNT_ROOT.iterdir()):
        MOUNT_ROOT = Path('/content/gdrive')
    drive.mount(str(MOUNT_ROOT), force_remount=False)
MYDRIVE = MOUNT_ROOT / 'MyDrive'
print('MyDrive:', MYDRIVE)

In [ ]:
# 2. 실험 브랜치 동기화
import subprocess, shutil

REPO_URL = 'https://github.com/tswaincae1221/SME_DATA.git'
REPO_BRANCH = 'agent/feature-dedup-ablation'
REPO_DIR = Path('/content/SME_DATA_feature_dedup')
if (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'fetch', 'origin', REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '-B', REPO_BRANCH, f'origin/{REPO_BRANCH}'], cwd=REPO_DIR, check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
print('repo:', REPO_DIR)

In [ ]:
# 3. 의존성
%pip install -q catboost==1.2.8
import sys, json
import pandas as pd
import torch, catboost
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'catboost', catboost.__version__)

In [ ]:
# 4. Drive 경로 — 폴더가 다르면 이 셀만 수정
DATA_ROOT = MYDRIVE / 'SME_DATA' / 'processed_station_features'
MASTER_CSV = DATA_ROOT / 'final_train_dataset_19to25_master.csv'
SHORTTERM_CSV = DATA_ROOT / 'shortterm_12to14_data' / 'incremental_12to14_tables' / 'shortterm_long_2019to2025.csv'
OUTPUT_ROOT = DATA_ROOT / 'model_experiments_19to25' / 'feature_dedup_ablation'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
for label, path in [('master', MASTER_CSV), ('shortterm', SHORTTERM_CSV)]:
    if not path.exists():
        raise FileNotFoundError(f'{label} 파일이 없습니다: {path}')
print(MASTER_CSV, SHORTTERM_CSV, OUTPUT_ROOT, sep='\n')

In [ ]:
# 5. 시드 42: 다섯 단계 전체 누적 실험
SCRIPT = REPO_DIR / 'scripts' / 'experiment_feature_dedup_ablation.py'
SEED42_DIR = OUTPUT_ROOT / 'seed42_all_steps'
cmd = [
    sys.executable, '-u', str(SCRIPT),
    '--master-csv', str(MASTER_CSV),
    '--shortterm-long-csv', str(SHORTTERM_CSV),
    '--output-dir', str(SEED42_DIR),
    '--seed', '42', '--device', 'auto', '--threads', '4',
]
print('$', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)

In [ ]:
# 6. 시드 43·44: 기준 모델과 단일 시드 최고였던 3단계 재검증
for seed in [43, 44]:
    output_dir = OUTPUT_ROOT / f'seed{seed}_step0_vs_step3'
    cmd = [
        sys.executable, '-u', str(SCRIPT),
        '--master-csv', str(MASTER_CSV),
        '--shortterm-long-csv', str(SHORTTERM_CSV),
        '--output-dir', str(output_dir),
        '--steps', '0', '3', '--seed', str(seed),
        '--device', 'auto', '--threads', '4',
    ]
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_DIR, check=True)

In [ ]:
# 7. 다중 시드 집계
parts = []
paths = {
    42: SEED42_DIR / 'cumulative_ablation_metrics.csv',
    43: OUTPUT_ROOT / 'seed43_step0_vs_step3' / 'cumulative_ablation_metrics.csv',
    44: OUTPUT_ROOT / 'seed44_step0_vs_step3' / 'cumulative_ablation_metrics.csv',
}
for seed, path in paths.items():
    frame = pd.read_csv(path)
    frame = frame[frame['step'].isin([0, 3])].copy()
    frame['seed'] = seed
    parts.append(frame)
runs = pd.concat(parts, ignore_index=True)
metric_columns = ['TA_RMSE', 'HM_RMSE', 'competition_score', 'TA_oof_RMSE', 'HM_oof_RMSE']
summary = runs.groupby(['step', 'variant'])[metric_columns].agg(['mean', 'std', 'min', 'max']).reset_index()
summary.columns = ['_'.join(str(x) for x in col if x != '') for col in summary.columns]
runs.to_csv(OUTPUT_ROOT / 'multiseed_step0_vs_step3_runs.csv', index=False)
summary.to_csv(OUTPUT_ROOT / 'multiseed_step0_vs_step3_summary.csv', index=False)
display(runs[['seed', 'step', 'TA_RMSE', 'HM_RMSE', 'competition_score']])
display(summary)

In [ ]:
# 8. 전체 단계 결과 확인
all_steps = pd.read_csv(SEED42_DIR / 'cumulative_ablation_metrics.csv')
display(all_steps[[
    'step', 'variant', 'TA_RMSE', 'HM_RMSE', 'competition_score',
    'delta_previous_competition_score', 'delta_baseline_competition_score',
]])
best = all_steps.sort_values('competition_score').iloc[0]
print('단일 시드 최저 점수:', best['variant'], best['competition_score'])
print('결과 저장:', OUTPUT_ROOT)